# SDR-Former — Siamese Dual-Resolution Transformer, tái lập từ bài báo

Huấn luyện **SDR-Former** phân loại **7 lớp** tổn thương gan trên volume **8 pha** MRI.

Nguồn: Lou, Ying, Liu, Zhou, Zhang, Yu, *SDR-Former: A Siamese Dual-Resolution Transformer
for Liver Lesion Classification Using 3D Multi-Phase Imaging*, arXiv:2402.17246 (2024).
Nhóm này **chính là nhóm phát hành LLD-MMRI**. Mốc của họ trên nhánh MR 8 pha:
macro-F1 **0.7910**, κ **0.7467**.

Cấu hình: `configs/sdrformer.yaml`. Không sửa gì trong notebook — mọi siêu tham số đọc từ
file đó.

## ⭐ Vì sao chạy cái này: nó đổi trục THỨ BA

UniFormer-Base (dung lượng lớn hơn) và UniFormerV2-B/16 (nguồn pretrain khác) đều **không**
vượt được cấu hình chính. Hai trục đó đã cạn. Cấu hình này đổi trục **"8 pha được kết hợp
thế nào"** — thứ dự án chưa từng thử: mọi thí nghiệm cho tới nay đều đưa 8 pha vào làm 8
**kênh** của conv đầu tiên (image-level fusion).

SDR-Former dùng **Siamese**: một encoder *dùng chung* chạy riêng cho từng pha, rồi hợp nhất
bằng một module attention học được (APSM). Bảng 1 của họ đo đúng trục đó, **một biến, trên
sáu backbone**:

| backbone | image-level | Siamese | hiệu |
|---|---|---|---|
| ResNet-50 | 0.6898 | 0.7168 | +0.027 |
| DenseNet-121 | 0.7171 | 0.7394 | +0.022 |
| MCSCNN | 0.7089 | 0.7409 | +0.032 |
| BoTNet-50 | 0.7139 | 0.7572 | +0.043 |
| UniFormer-S | 0.7123 | 0.7639 | **+0.052** |
| H2Former | 0.7342 | 0.7745 | +0.040 |

**Sáu trên sáu đều dương**, trung bình +0.036, transformer hưởng lợi hơn CNN.

## ⚠️⚠️ Đánh đổi phải biết TRƯỚC khi đọc kết quả: cấu hình này BỎ pretrained

SDR-Former train **from scratch** — không có checkpoint công khai cho DR-Former. Mà can
thiệp duy nhất từng thắng có ý nghĩa thống kê trong dự án này chính là pretrained Kinetics
(+0.130, P < 0.001).

| | macro-F1 |
|---|---|
| UniFormer-S from scratch, recipe ban tổ chức | 0.6083 |
| UniFormer-S from scratch, recipe của họ | 0.7123 |
| SDR-Former from scratch, recipe của họ | **0.7910** |
| **ta: UniFormer-S + Kinetics** | **0.7682** test-104 · **0.8147** out-of-fold |

0.7910 chỉ hơn ta **+0.023** trên test, nằm gọn trong khoảng tin cậy ±0.09. **Đây không phải
một cấu hình chắc chắn tốt hơn** — nó là phép thử trục fusion, giá trị chính nằm ở chỗ đó.

## Cần mount gì

| | |
|---|---|
| **Cache lưới `128×128×16`** | Sinh bởi `configs/preprocess_cghnet.yaml` — **dùng lại**, không build mới |
| **Internet** | Không cần. Model train from scratch, không tải trọng số nào |

§4.2 của bài: *"uniformly resized to 16 × 128 × 128 ... randomly crop to 14 × 112 × 112"* —
khớp **chính xác** cache đã có.

## Sáu cổng chạy TRƯỚC khi cam kết fold nào

Mỗi cổng chặn một chế độ hỏng **im lặng** — loại lỗi vẫn chạy trơn và vẫn ra số hợp lý.

| cổng | chặn gì |
|---|---|
| **A** tham số | bản tái lập lệch kiến trúc so với bài mà không ai thấy |
| **B** hình học | hai nhánh lệch tỉ lệ ⇒ BCIM ghép sai, **không nổ** |
| **C** ngân sách | phát hiện chi phí quá cao **sau khi** đã cam kết cả session |
| **D** APSM | module chọn pha không thật sự chọn gì (trọng số phẳng đều) |
| **E** augment | augmentation phá cấu trúc đa pha |
| **F** Siamese | encoder **không** thật sự dùng chung trọng số giữa 8 pha |

Chạy tuần tự từ trên xuống. Cổng nào đỏ thì **dừng**, đừng chạy mục 2.


## 0. Bootstrap

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/hdtruong802/liver-mri-3d-classifier.git"

# ---- THAM SỐ ---------------------------------------------------------------
FOLDS = [1]                    # ⚠️ 1 fold KHÔNG kết luận được gì (CI ~±0.19). Nó chỉ
                               # dùng để LOẠI, và chỉ khi thấp hẳn. Xem bar ở mục 3.
CONFIG_NAME = "sdrformer.yaml"
PREPROCESS_NAME = "preprocess_cghnet.yaml"
# ----------------------------------------------------------------------------

REPO = Path("/kaggle/working/repo")
os.chdir("/kaggle/working")
subprocess.run(["rm", "-rf", str(REPO)], check=False)
subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO)], check=True)
sys.path.insert(0, str(REPO))
os.chdir(REPO)

for name in [m for m in list(sys.modules) if m == "src" or m.startswith("src.")]:
    del sys.modules[name]

print("repo commit:", subprocess.run(
    ["git", "-C", str(REPO), "log", "-1", "--format=%h %s"],
    capture_output=True, text=True,
).stdout.strip())

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "monai"], check=True)

EXPERIMENT = Path(CONFIG_NAME).stem
os.environ["LLDMMRI_OUTPUT_DIR"] = f"/kaggle/working/runs/{EXPERIMENT}"
os.environ.pop("LLDMMRI_DATA_ROOT", None)

from src.utils.io import load_yaml, repo_root  # noqa: E402

assert repo_root() == REPO.resolve(), "src/ nạp từ chỗ khác — restart kernel"

CFG_PATH = REPO / "configs" / CONFIG_NAME
CFG = load_yaml(CFG_PATH)
PRE = load_yaml(REPO / "configs" / PREPROCESS_NAME)
M = CFG["model"]

INNER = tuple(PRE["target_size"])                       # [X, Y, Z] = 112, 112, 14
MARGIN = tuple(PRE["crop_margin_voxels"])
GRID = tuple(s + 2 * m for s, m in zip(INNER, MARGIN))  # 128, 128, 16
assert tuple(CFG["data"]["crop_size"]) == INNER, "data.crop_size lệch target_size của cache"

# Thứ tự của MODEL là (D, H, W) với D = trục lát; cache là [X, Y, Z]. Đổi một lần ở đây và
# dùng biến này ở mọi cổng, để không ai phải nhẩm lại giữa chừng.
MODEL_SIZE = (INNER[2], INNER[0], INNER[1])             # (14, 112, 112)

print(f"\nthí nghiệm: {EXPERIMENT} · fold {FOLDS}")
print(f"model     : sdrformer · stem {M['stem_channels']} · stage {M['stage_channels']}")
print(f"            blocks/stage {M['blocks_per_stage']} · bcim_hidden_mult "
      f"{M['bcim_hidden_mult']} · heads {M['num_heads']}")
print(f"            attention {M['attention']} lưới {M['grid_size']} · "
      f"BCIM {M['use_bcim']} · APSM {M['use_apsm']}")
print(f"loss      : {CFG['loss']['name']} · class_weights {CFG['loss']['class_weights']} · "
      f"smoothing {CFG['loss']['label_smoothing']}")
print(f"train     : {CFG['train']['epochs']} epoch · lr {CFG['train']['lr']} · "
      f"wd {CFG['train']['weight_decay']} · batch {CFG['data']['batch_size']}")
print(f"sampler   : {CFG['data']['sampling']}")
print(f"hình học  : cache {GRID} -> model nhận {INNER} [X,Y,Z] = {MODEL_SIZE} (D,H,W)")

# Bài dùng 200 epoch, mọi config khác của dự án là 300. Rất dễ bị "thống nhất" nhầm.
assert CFG["train"]["epochs"] == 200, "§4.2 của bài là 200 epoch, không phải 300"
assert CFG["loss"]["name"] == "cross_entropy", "§4.2: standard cross-entropy, không phải focal"

## 1. Cache

Cần cache có lưới **`128×128×16`** (`configs/preprocess_cghnet.yaml`): mô hình nhận
`112×112×14`, phần dư mỗi phía là lề cho phép cắt ngẫu nhiên lúc train và cắt giữa lúc suy luận.

⚠️ Cache hình học khác **không dùng được** — `data.crop_size` phải khớp `target_size` của
chính cache đang mount, và cell dưới `assert` điều đó trước khi chạy tiếp.

In [ ]:
import json as _json

import numpy as np

CAN = {
    "align_phases": "per_phase",
    "crop_mode": "lesion_tight",
    "target_size": list(INNER),
    "crop_margin_voxels": list(MARGIN),
}

ung_vien, CACHE_DIR = [], None
for meta_path in sorted(Path("/kaggle/input").rglob("cache_meta.json")):
    try:
        meta = _json.loads(meta_path.read_text("utf-8"))
    except Exception:  # noqa: BLE001 - chỉ để liệt kê chẩn đoán
        continue
    khop = all(meta.get(k) == v for k, v in CAN.items())
    ung_vien.append((meta_path.parent, meta, khop))
    if khop and CACHE_DIR is None:
        CACHE_DIR = meta_path.parent

print(f"=== {len(ung_vien)} cache tìm thấy dưới /kaggle/input ===")
for path, meta, khop in ung_vien:
    print(f"  {'✓ khớp  ' if khop else '  --    '}  {path}")
    print(f"          size={meta.get('target_size')} lề={meta.get('crop_margin_voxels')} "
          f"align={meta.get('align_phases')} crop={meta.get('crop_mode')}")

if CACHE_DIR is None:
    raise RuntimeError(
        "Chưa mount cache đúng hình học.\n"
        f"  Cần cache có {CAN}\n"
        "  Chạy notebooks/18_build_cache_cghnet.ipynb trước (CPU, Accelerator = None,\n"
        "  ~20 phút), lưu output thành Dataset, rồi mount vào đây.\n"
        "  Cache lưới khác KHÔNG dùng được — bảng ở trên liệt kê mọi cache đang mount."
    )

os.environ["LLDMMRI_CACHE_DIR"] = str(CACHE_DIR)

n_npz = len(list(CACHE_DIR.glob("*.npz")))
assert n_npz >= 498, f"chỉ có {n_npz} ca, cần 498"
with np.load(next(CACHE_DIR.glob("*.npz"))) as z:
    shape = tuple(z["image"].shape)
assert shape == (8, *GRID), f"mảng {shape}, cần {(8, *GRID)} — cache này sai hình học"
print(f"\ncache ✓ · {CACHE_DIR} · {n_npz} ca · mảng {shape}")

## 1b. Không có trọng số pretrained — và đó là điều phải ghi vào báo cáo

SDR-Former train **from scratch**. Không có checkpoint công khai nào cho DR-Former, nên
notebook này **không tải gì** và **không cần Internet**.

⚠️ Nghĩa là cấu hình này cố ý bỏ đòn bẩy duy nhất từng thắng có ý nghĩa thống kê trong dự
án (+0.130 từ Kinetics). Nếu kết quả thấp hơn cấu hình chính thì **không kết luận được**
"Siamese fusion không có tác dụng" — vì phép so gộp hai biến: fusion *và* pretrained.

Kết luận sạch duy nhất rút ra được từ một kết quả thấp là: *ở điều kiện from-scratch, trục
fusion không bù được phần mất đi do bỏ pretrained.*


## Cổng A ⚠️⚠️ — bản tái lập có đúng kích thước như bài không

Bài **không công khai code**, nên đây là tái lập từ văn bản. Neo duy nhất kiểm được là
**số tham số** mà Bảng 4 của họ công bố: **19.34M** / **40.26 GFLOPs** cho bản 8 pha.

⚠️ **Bản dựng literal theo Hình 2 cho ~12.7M, tức THIẾU ~6.6M so với bài.** Bài không cho
đủ thông tin để biết thiếu ở đâu. Hai chỗ khả dĩ nhất, đã tính sẵn:

| khoá | thêm | căn cứ |
|---|---|---|
| `blocks_per_stage: 2` | ~+1.2M | Hình 2 vẽ **một** hộp, nhưng hộp thường ký hiệu nhóm lặp |
| `bcim_hidden_mult: 2` | ~+3.5M | `W²₃ₓ₃ₓ₃` đi `2C → 2C → C` thay vì `2C → C → C` |

Bật cả hai cho ~17.3M, **vẫn chưa đúng 19.34M**.

**Giữ bản literal là quyết định có lý do khoa học**, không phải cho tiện: dự án vừa đo được
rằng *tăng dung lượng làm tệ đi* trên 312 ca train (UniFormer-Base thua UniFormer-S). Bản
nhỏ hơn bài vì thế **hợp** với chẩn đoán hiện tại.

⚠️ Nhưng mọi câu trong báo cáo dùng số của cấu hình này **bắt buộc** ghi rằng đây là bản tái
lập **nhỏ hơn** bài, và 0.7910 chỉ là mốc định hướng.

⚠️ Bảng 4 còn một chỗ tự nó đã lạ: bản **3 pha** của họ có **nhiều** tham số hơn bản 8 pha
(28.52M so với 19.34M) và nhiều FLOPs hơn (102.30 so với 40.26). Cùng kiến trúc mà thêm pha
lại rẻ đi là không thể ⇒ gần như chắc chắn họ dùng **hai cấu hình kích thước khác nhau** cho
hai dataset, và bài không nói cấu hình nào.


In [ ]:
import torch

from src.models import build_model
from src.models.sdrformer import PAPER_FLOPS_G, PAPER_MR, PAPER_PARAMS_M, SNN_GAIN_MR

model = build_model(M)

n_tong = sum(p.numel() for p in model.parameters())
n_enc = sum(p.numel() for p in model.encoder.parameters())
n_apsm = sum(p.numel() for p in model.apsm_c.parameters()) + sum(
    p.numel() for p in model.apsm_v.parameters()
)
n_bcim = sum(
    p.numel() for b in model.encoder.bcims if b is not None for p in b.parameters()
)

print(f"{'tham số tổng':<22}{n_tong / 1e6:>8.2f}M      (bài, Bảng 4: {PAPER_PARAMS_M}M)")
print(f"{'  encoder dùng chung':<22}{n_enc / 1e6:>8.2f}M")
print(f"{'    trong đó BCIM':<22}{n_bcim / 1e6:>8.2f}M")
print(f"{'  APSM (x2)':<22}{n_apsm / 1e6:>8.2f}M")
print(f"{'  còn lại':<22}{(n_tong - n_enc - n_apsm) / 1e6:>8.2f}M")
print(f"\nGFLOPs bài công bố    : {PAPER_FLOPS_G}  (không đo ở đây)")

lech = n_tong / 1e6 - PAPER_PARAMS_M
print(f"\nlệch so với bài       : {lech:+.2f}M  ({100 * lech / PAPER_PARAMS_M:+.0f}%)")
print("\n⚠️ Lệch âm là ĐÃ BIẾT và ĐÃ CHỌN — xem markdown ở trên. Không phải lỗi.")
print("   Muốn tiến gần 19.34M: blocks_per_stage 2 và/hoặc bcim_hidden_mult 2.")

# Chặn chiều NGƯỢC lại: nếu ai đó vô tình dựng ra một mạng LỚN hơn bài thì đó là lỗi thật —
# bản literal không có đường nào vượt 19.34M.
assert n_tong / 1e6 < PAPER_PARAMS_M, (
    f"⛔ {n_tong / 1e6:.2f}M VƯỢT {PAPER_PARAMS_M}M của bài — bản literal không thể lớn hơn. "
    "Ai đó đã đổi stage_channels/blocks_per_stage."
)
# Và chặn ngưỡng dưới: dưới 8M nghĩa là một nhánh nào đó không được dựng.
assert n_tong / 1e6 > 8.0, f"⛔ chỉ {n_tong / 1e6:.2f}M — quá nhỏ, nghi thiếu hẳn một nhánh"

print(f"\nmốc đối chiếu của bài (MR 8 pha): macro-F1 {PAPER_MR['f1']} · κ {PAPER_MR['kappa']}")
print(f"hiệu Siamese trên 6 backbone   : "
      f"{min(s - p for p, s in SNN_GAIN_MR.values()):+.3f} .. "
      f"{max(s - p for p, s in SNN_GAIN_MR.values()):+.3f}")
print("\ncổng A ✓")

## Cổng B ⚠️⚠️⚠️ — hai nhánh có đúng tỉ lệ không

Kiến trúc này có một bất biến mà **nếu sai thì không có gì nổ**: nhánh CNN phải có **đúng
2× số voxel trong mặt phẳng** và **cùng số lát** với nhánh Transformer, ở *mọi* stage. Bài
định nghĩa `F_v` là `C × D × H/2 × W/2`.

Sai tỉ lệ thì `BCIM` vẫn chạy — `F.interpolate` nhận mọi kích thước — và chỉ lặng lẽ ghép
hai feature map lệch nhau. Model vẫn hội tụ, vẫn ra số trông hợp lý, chỉ thấp hơn đáng lẽ.

Cell dưới bắt hook vào **từng BCIM** để đọc shape **thật** của cả hai nhánh, rồi đối chiếu
với `stage_shapes` tính bằng tay.

Đối chiếu Hình 2 của bài (họ vẽ ở 16 lát trước khi cắt; ta cắt còn 14):

```
bài:  CNN   16×16×56×56 → 32×16×28×28 → 64×8×14×14 → 128×8×14×14
ta :  CNN   16×14×56×56 → 32×14×28×28 → 64×7×14×14 → 128×7×14×14
bài:  Trans 16×16×28×28 → 32×16×14×14 → 64×8×7×7   → 128×8×7×7
ta :  Trans 16×14×28×28 → 32×14×14×14 → 64×7×7×7   → 128×7×7×7
```


In [ ]:
from src.models.sdrformer import DEFAULT_GRID, stage_shapes

thuc_te = []
hooks = [
    b.register_forward_hook(
        lambda _m, inp, _out: thuc_te.append((tuple(inp[0].shape[1:]), tuple(inp[1].shape[1:])))
    )
    for b in model.encoder.bcims
    if b is not None
]
assert hooks, "⛔ không có BCIM nào — model.use_bcim đang False?"

_dev_b = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(_dev_b).eval()
with torch.no_grad():
    ra = model(torch.zeros(1, 8, *INNER, device=_dev_b))
for h in hooks:
    h.remove()
model.cpu()
ra = ra.cpu()

tinh_tay = stage_shapes(MODEL_SIZE)

print(f"{'stage':<9}{'kênh':>6}{'CNN (thật)':>20}{'Trans (thật)':>18}{'tỉ lệ':>9}")
for (ten, ch, cnn_tay, tr_tay), (cnn_that, tr_that) in zip(tinh_tay[1:], thuc_te):
    # inp của BCIM là feature map TRƯỚC pool của stage đó; stage_shapes ghi SAU pool.
    ti = f"{cnn_that[2] / tr_that[2]:.1f}x"
    print(f"{ten:<9}{ch:>6}{str(cnn_that):>20}{str(tr_that):>18}{ti:>9}")
    assert cnn_that[0] == tr_that[0], f"⛔ {ten}: số kênh lệch"
    assert cnn_that[1] == tr_that[1], f"⛔ {ten}: SỐ LÁT lệch {cnn_that[1]} vs {tr_that[1]}"
    assert cnn_that[2] == 2 * tr_that[2], f"⛔ {ten}: trục H không gấp đôi"
    assert cnn_that[3] == 2 * tr_that[3], f"⛔ {ten}: trục W không gấp đôi"

print(f"\n{'stage':<9}{'kênh':>6}{'CNN (sau pool)':>20}{'Trans (sau pool)':>20}")
for ten, ch, cnn, tr in tinh_tay:
    print(f"{ten:<9}{ch:>6}{str(cnn):>20}{str(tr):>20}")

gd, gh, gw = DEFAULT_GRID
print(f"\nlưới GSA {tuple(DEFAULT_GRID)} ⇒ {gd * gh * gw} token mỗi nhóm attention, ở MỌI stage")
print("   (so với 2744 token attention toàn cục của cấu hình chính — rẻ hơn rất nhiều)")

assert tuple(ra.shape) == (1, M["num_classes"]), f"⛔ đầu ra {tuple(ra.shape)}"
print(f"\nđầu ra: {tuple(ra.shape)}")
print("\ncổng B ✓ — hai nhánh đúng tỉ lệ 2× trong mặt phẳng, cùng số lát")

## Cổng C ⚠️ — ngân sách, ĐO THẬT

Phải **đo** s/epoch, không được suy từ GFLOPs: ước lượng kiểu đó cho CGHNet đã sai xa
(WORKLOG S-123), và cho UniFormer-Base thì sai cả về **bộ nhớ** chứ không chỉ thời gian.

⚠️ **Nút thắt của kiến trúc này là Siamese, không phải attention.** Encoder chạy `B × 8`
lượt thay vì `B` lượt, nên với `batch_size: 8` thì mỗi bước có **64 volume** đi qua encoder.
Bù lại encoder rất nhỏ (~12.7M) và mỗi nhóm attention chỉ 98 token.

Bài dùng 200 epoch (không phải 300), nên ngân sách rộng hơn cấu hình chính.

Quá **60 s/epoch** ⇒ 5 fold không lọt. Khoá thoát theo thứ tự ưu tiên:
1. `data.batch_size: 4` + `train.accum_steps: 2` — giữ batch hiệu dụng, giảm đỉnh bộ nhớ
   (⚠️ BatchNorm sẽ thấy 32 volume thay vì 64, phải ghi là chỗ lệch);
2. `model.grid_size: [2, 4, 4]` — ít token hơn mỗi nhóm.


In [ ]:
import time

from src.train.run import build_loaders

train_loader, val_loader, train_labels = build_loaders(CFG, FOLDS[0])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device).train()
opt = torch.optim.AdamW(model.parameters(), lr=float(CFG["train"]["lr"]))
scaler = torch.amp.GradScaler("cuda", enabled=bool(CFG["train"]["amp"]))

N_DO = 8
t0 = None
for i, batch in enumerate(train_loader):
    if i == 2:          # bỏ 2 batch đầu: nạp worker + cudnn benchmark
        torch.cuda.synchronize() if device.type == "cuda" else None
        t0 = time.perf_counter()
    if i == 2 + N_DO:
        break
    x = batch["image"].to(device, non_blocking=True)
    y = batch["label"].to(device, non_blocking=True)
    with torch.amp.autocast("cuda", enabled=bool(CFG["train"]["amp"])):
        loss = torch.nn.functional.cross_entropy(model(x), y)
    opt.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.step(opt)
    scaler.update()
if device.type == "cuda":
    torch.cuda.synchronize()

assert t0 is not None, (
    f"⛔ chỉ có {len(train_loader)} batch, cần > 2 để bỏ warm-up. Kiểm lại batch_size."
)
giay_moi_batch = (time.perf_counter() - t0) / N_DO
n_batch = len(train_loader)
s_epoch = giay_moi_batch * n_batch * 1.15      # +15% cho vòng val, ước từ các run trước
gio_fold = s_epoch * int(CFG["train"]["epochs"]) / 3600

print(f"thiết bị      : {device}")
print(f"batch train   : {n_batch} · batch_size {CFG['data']['batch_size']}")
print(f"giây/batch    : {giay_moi_batch:.3f}")
print(f"giây/epoch    : {s_epoch:.1f}   (đã cộng ~15% cho val)")
print(f"giờ/fold      : {gio_fold:.2f}  ({CFG['train']['epochs']} epoch)")
print(f"giờ/{len(FOLDS)} fold  : {gio_fold * len(FOLDS):.2f}")
print(f"giờ/5 fold    : {gio_fold * 5:.2f}")
print("\nràng buộc Kaggle: session tối đa 12h · quota GPU ~30h/tuần")
print(f"volume qua encoder mỗi bước: {CFG['data']['batch_size']} x 8 pha = "
      f"{CFG['data']['batch_size'] * 8}")
if device.type == "cuda":
    print(f"đỉnh VRAM     : {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

if s_epoch > 60:
    print(f"\n⛔ {s_epoch:.0f} s/epoch — QUÁ NGƯỠNG 60. Hai khoá thoát, theo thứ tự:")
    print("   1. data.batch_size 4 + train.accum_steps 2  (giữ batch hiệu dụng 8;")
    print("      ⚠️ BatchNorm thấy 32 volume thay vì 64 — ghi là chỗ lệch)")
    print("   2. model.grid_size [2, 4, 4]  (32 token/nhóm thay vì 98)")
    print("   Rồi chạy lại từ cổng A.")
else:
    print("\ncổng C ✓")

model.cpu()
del opt, scaler
torch.cuda.empty_cache() if device.type == "cuda" else None

## Cổng D ⚠️ — APSM có thật sự CHỌN pha không

APSM là đóng góp mà bài gán mức tăng lớn nhất trên dataset MR (Bảng 3: `Baseline` 0.7508 →
`+APSM` 0.7636 → đủ bộ 0.7910), và họ giải thích rằng nó ăn hơn trên MR *vì MR có 8 pha còn
CT chỉ có 3*.

Nhưng một softmax **chạy sai trục** vẫn cho ra tensor đúng shape, vẫn hội tụ, và chỉ lặng lẽ
biến APSM thành một phép trung bình đắt tiền. Cell này kiểm ba điều:

* **D1** — tổng trọng số trên **trục pha** bằng 1 cho từng kênh (Eq. 2). Nếu softmax chạy
  nhầm trục kênh thì tổng theo pha sẽ **không** bằng 1.
* **D2** — trọng số **không phẳng đều**. Bằng `1/8` ở mọi pha nghĩa là module chưa học được
  gì (ở khởi tạo thì điều đó *bình thường*, nên đây chỉ in ra để đọc, không assert).
* **D3** — trọng số **đổi theo đầu vào**. Hai ca khác nhau phải cho phân bố pha khác nhau;
  giống hệt nhau nghĩa là APSM đang bỏ qua đầu vào.


In [ ]:
# Tên 8 pha nằm ở configs/data.yaml (không phải taxonomy.py — chỗ đó là 7 LỚP tổn thương).
PHASE_NAMES = [p["name"] for p in load_yaml(REPO / "configs" / "data.yaml")["phases"]]
assert len(PHASE_NAMES) == M["num_phases"], (
    f"⛔ configs/data.yaml khai {len(PHASE_NAMES)} pha nhưng model cần {M['num_phases']}"
)

if not M["use_apsm"]:
    print("cổng D — BỎ QUA: use_apsm=False (đang chạy ablation Bảng 3)")
else:
    model.to(_dev_b).eval()
    with torch.no_grad():
        model(torch.randn(2, 8, *INNER, device=_dev_b))
    w = model.apsm_c.last_weights.float().cpu()          # [B, P, C]
    print(f"trọng số APSM: {tuple(w.shape)}  (batch, pha, kênh)")

    # --- D1: softmax đúng trục -------------------------------------------------
    tong = w.sum(dim=1)
    assert torch.allclose(tong, torch.ones_like(tong), atol=1e-4), (
        f"⛔ tổng trên trục PHA = {tong.mean():.4f}, phải là 1.0 — softmax đang chạy sai trục. "
        "Eq. (2) của bài softmax trên trục pha, riêng cho từng kênh."
    )
    print(f"  D1 ✓ tổng trên trục pha = 1.000 cho cả {w.shape[0] * w.shape[2]} (ảnh, kênh)")

    # --- D2: có phân biệt các pha không ---------------------------------------
    tb = w[0].mean(dim=1)                                 # [P] — trung bình trên kênh
    deu = 1.0 / w.shape[1]
    print(f"\n  {'pha':<12}{'trọng số TB':>13}{'so mức đều':>13}")
    for i, ten in enumerate(PHASE_NAMES):
        print(f"  {ten:<12}{tb[i]:>13.4f}{tb[i] / deu:>12.2f}x")
    print(f"  mức đều = 1/8 = {deu:.4f} · độ lệch chuẩn giữa các pha = {tb.std():.4f}")
    if tb.std() < 1e-3:
        print("  ⚠ trọng số gần như PHẲNG ĐỀU. Ở model CHƯA TRAIN điều này là bình thường")
        print("    (8 conv 1x1x1 khởi tạo gần giống nhau). Chạy lại cổng này SAU khi train:")
        print("    phẳng đều ở model đã train nghĩa là APSM không học được gì.")
    else:
        print("  D2 ✓ có phân biệt giữa các pha")

    # --- D3: có phụ thuộc đầu vào không ---------------------------------------
    lech = (w[0] - w[1]).abs().max()
    assert lech > 1e-6, (
        "⛔ hai ảnh khác nhau cho trọng số GIỐNG HỆT nhau — APSM đang bỏ qua đầu vào. "
        "Kiểm lại `reduce` có nhận feature map thật không."
    )
    print(f"\n  D3 ✓ trọng số đổi theo đầu vào (lệch tối đa giữa 2 ảnh: {lech:.2e})")
    print("\ncổng D ✓")

model.cpu()

## Cổng E ⚠️ — augmentation có phá cấu trúc đa pha không

Chẩn đoán u gan trên MRI đa pha dựa vào cường độ **tương đối giữa các pha**. Một
augmentation vẽ **tham số ngẫu nhiên riêng cho từng pha** sẽ đổ nhiễu thẳng lên chính tín
hiệu phân biệt, mà vẫn cho ảnh trông bình thường. E6 đã đo tác hại đó: ICC −0.085, di căn
−0.111 (WORKLOG S-102).

Cấu hình này **tắt cả ba augment lọc không gian** (`edge`/`emboss`/`filter`) vì chúng thuộc
recipe đội hạng 2, **không** có trong bài SDR-Former — bật chúng là trộn hai recipe và mốc
0.7910 mất nghĩa. Nên chuỗi transform ở đây chỉ còn hình học.

**E1** kiểm bất biến "cùng tham số cho cả 8 pha".
**E2** kiểm phép xoay có lấp giá trị 0 vào góc không — đó là lệch phân bố train/val có hệ
thống ở *mọi* bước huấn luyện (WORKLOG S-111).

⚠️ **Chỗ thiếu đã biết:** §4.2 của bài liệt kê *"random rotations, erasing, and flips"*. Dự
án không có `RandomErasing3D` và lần này không cài thêm. **Phải ghi vào báo cáo.**


In [ ]:
import numpy as np

from src.data.transforms import RandomAppearance, build_train_transform

AUG = CFG["data"]["augment"]
chain = build_train_transform(AUG, CFG["data"]["crop_size"])
print("chuỗi transform:", [type(t).__name__ for t in chain.transforms])

# Config này tắt cả ba augment lọc, nên KHÔNG được có RandomAppearance nào hoạt động.
app = [t for t in chain.transforms if isinstance(t, RandomAppearance)]
for khoa in ("edge_prob", "emboss_prob", "filter_prob"):
    assert AUG[khoa] == 0, (
        f"⛔ {khoa}={AUG[khoa]} — ba augment lọc thuộc recipe đội hạng 2, KHÔNG có trong bài "
        "SDR-Former. Bật chúng là trộn hai recipe."
    )
print(f"ba augment lọc: TẮT ✓ (đúng bài) · RandomAppearance trong chuỗi: {len(app)}")

# --- E1: cùng tham số cho cả 8 pha ------------------------------------------
from src.utils.seed import set_seed

set_seed(1337)
lech = 0
for _ in range(200):
    goc = torch.randn(8, *GRID)
    goc[3] = goc[0]                       # pha 3 sao chép pha 0
    ra = chain({"image": goc.clone()})["image"]
    if not torch.allclose(ra[0], ra[3], atol=1e-4):
        lech += 1
print(f"\nE1 · 200 lượt, {lech} lượt pha 0 khác pha 3")
assert lech == 0, (
    f"⛔ {lech} lượt áp KHÁC NHAU giữa các pha. Có tham số ngẫu nhiên vẽ riêng cho từng pha "
    "— augmentation đang phá cấu trúc đa pha. DỪNG."
)
print("  ✓ cùng tham số hình học cho cả 8 pha")

# --- E2: rìa train so với val -----------------------------------------------
# Dùng LẠI loader của cổng C. Mỗi loader giữ 4 worker process; dựng thêm một bộ nữa là ăn
# RAM của Kaggle vô ích, và cổng C đã dựng đúng cặp cần so (train có augment, val thì không).
assert "train_loader" in globals(), "⛔ chạy cổng C trước — cổng E dùng lại loader của nó"


def ti_le_0_o_ria(img):
    """img: [B, 8, X, Y, Z] — tỉ lệ voxel ~0 trong dải 4 voxel sát rìa mặt phẳng."""
    ria = torch.cat([img[:, :, :4].flatten(), img[:, :, -4:].flatten(),
                     img[:, :, :, :4].flatten(), img[:, :, :, -4:].flatten()])
    return float((ria.abs() < 1e-6).float().mean())


def do_ria(loader, n_batch=6):
    vals = []
    for i, b in enumerate(loader):
        if i >= n_batch:
            break
        vals.append(ti_le_0_o_ria(b["image"]))
    return float(np.mean(vals))


tr, va = do_ria(train_loader), do_ria(val_loader)
print(f"\nE2 · voxel 0 ở rìa: train {tr:.4f} · val {va:.4f} · lệch {abs(tr - va):.4f}")
assert abs(tr - va) < 0.02, (
    "⛔ lệch phân bố rìa train/val — kiểm `rotate_mode` có đúng `nearest` không."
)
print("  ✓ không có dải đệm 0 lệch giữa train và val")
print("\n⚠️ CHỖ THIẾU đã biết so với bài: `random erasing` (§4.2). Ghi vào báo cáo.")
print("\ncổng E ✓")

## Cổng F ⚠️⚠️ — encoder có THẬT SỰ dùng chung trọng số không

Đây là cổng **riêng của kiến trúc này**, và nó chặn một chế độ hỏng vừa im lặng vừa đắt.

Toàn bộ tính khả thi của hướng Siamese nằm ở chỗ **8 pha dùng chung một bộ tham số**: số
tham số giữ nguyên như một encoder đơn, chỉ chi phí *tính toán* nhân lên. Với 312 ca train,
tám encoder riêng gần như chắc chắn overfit — và đó chính là điều bài nói ở §3.1 khi chê
feature-level fusion.

Nếu ai đó (hoặc một lần refactor) vô tình tạo 8 encoder riêng thì: model vẫn chạy, vẫn hội
tụ, số tham số phồng lên ~8 lần mà **không có gì báo**, và kết quả sẽ thấp — rồi bị đọc
nhầm thành "Siamese fusion không có tác dụng".

Hai phép kiểm:

* **F1** — số tham số của encoder **không đổi** khi đổi `num_phases`. Đây là phép kiểm tất
  định, không phụ thuộc dữ liệu.
* **F2** — đưa **cùng một ảnh** vào hai vị trí pha khác nhau thì encoder phải trả về **đặc
  trưng giống hệt**. Trọng số dùng chung ⇒ hàm áp lên mỗi pha là *một* hàm.


In [ ]:
from src.models.sdrformer import build_sdrformer

# --- F1: số tham số encoder không phụ thuộc số pha --------------------------
tham_so_khac = {k: v for k, v in M.items() if k not in ("name", "num_phases")}
enc = {}
for p in (2, 8):
    m = build_sdrformer(num_phases=p, **tham_so_khac)
    enc[p] = sum(q.numel() for q in m.encoder.parameters())
    del m
print(f"tham số encoder · 2 pha: {enc[2]:,}")
print(f"tham số encoder · 8 pha: {enc[8]:,}")
assert enc[2] == enc[8], (
    f"⛔ encoder KHÔNG dùng chung trọng số: {enc[2]:,} so với {enc[8]:,}. Toàn bộ lập luận "
    "về tính khả thi của hướng này sụp đổ — 8 encoder riêng trên 312 ca train sẽ overfit."
)
print(f"  F1 ✓ giống hệt nhau ⇒ MỘT bộ tham số dùng chung cho cả 8 pha")

# --- F2: cùng ảnh ở hai vị trí pha -> cùng đặc trưng ------------------------
model.eval()
x = torch.randn(1, 8, *INNER)
x[:, 5] = x[:, 1]                     # pha 5 sao chép pha 1
with torch.no_grad():
    # Chạy thẳng encoder trên từng pha, đúng cách `forward` gộp B*P.
    xp = x.permute(0, 1, 4, 2, 3).contiguous()
    fc, fv = model.encoder(xp.reshape(8, 1, *xp.shape[2:]))
d_c = (fc[1] - fc[5]).abs().max().item()
d_v = (fv[1] - fv[5]).abs().max().item()
print(f"\n  pha 1 so pha 5 (cùng ảnh): lệch nhánh CNN {d_c:.2e} · nhánh Trans {d_v:.2e}")
assert d_c < 1e-4 and d_v < 1e-4, (
    "⛔ cùng một ảnh ở hai vị trí pha cho đặc trưng KHÁC nhau — encoder không dùng chung, "
    "hoặc có tham số phụ thuộc chỉ số pha nằm bên trong encoder."
)
print("  F2 ✓ encoder là MỘT hàm, áp như nhau cho mọi pha")

# Và ghi lại con số để đọc: chi phí tính toán nhân lên bao nhiêu lần.
print(f"\n⚠️ Đánh đổi: tham số giữ nguyên, nhưng encoder chạy {M['num_phases']}x lượt mỗi")
print(f"   bước. Đó là lý do cổng C phải đo thật chứ không suy từ số tham số.")
print("\ncổng F ✓")

del enc
import gc

gc.collect()

## 2. Train

Sáu cổng phải xanh hết trước khi chạy cell này. `resume: true` nên ngắt session giữa chừng
thì chạy lại là đọc tiếp từ `last.pt`.

⚠️ **200 epoch**, không phải 300 — §4.2 của bài.


In [ ]:
from src.train.run import TRAIN_RESULT_KEYS, train

# Giải phóng model của cổng A–C và MỌI loader của các cổng. Mỗi loader giữ 4 worker
# process; để chúng sống suốt lúc train là ăn RAM và tranh CPU với worker thật.
for _ten in ("model", "train_loader", "val_loader", "chain", "app", "x", "fc", "fv"):
    globals().pop(_ten, None)
import gc

gc.collect()
torch.cuda.empty_cache()

print("khoá train() trả về:", TRAIN_RESULT_KEYS)
results = {}
for fold in FOLDS:
    print(f"\n{'=' * 70}\nfold {fold}\n{'=' * 70}")
    results[fold] = train(CFG_PATH, fold)
    # ⚠️ `best_macro_f1`, KHÔNG phải `macro_f1`. Đọc sai tên thì `KeyError` nổ ở đúng dòng này,
    # tức SAU KHI đã train xong cả fold, và vòng lặp dừng luôn trước fold kế tiếp.
    print("fold %d xong: macro-F1 %.4f @ epoch %d"
          % (fold, results[fold]["best_macro_f1"], results[fold]["best_epoch"]))

## 3. Kết quả từng fold, và bar quyết định đã chốt trước

⚠️ **`/kaggle/working` bị xoá khi session kết thúc.** Chạy mục này ở session khác với session
train thì thư mục run không còn — cell **sẽ nổ kèm hướng dẫn** thay vì in một bảng rỗng.

### Bar đã chốt TRƯỚC khi chạy (fold 1, so với fold 1 của cấu hình chính = 0.8111)

| gộp fold 1 | kết luận |
|---|---|
| **≥ 0.78** | trục fusion có giá trị **dù mất pretrained** ⇒ chạy đủ 5 fold |
| **0.72–0.78** | có tác dụng nhưng không bù được pretrained ⇒ cân nhắc, ưu tiên deliverable |
| **0.65–0.72** | ngang mức from-scratch tốt của họ (0.7123) ⇒ **kết quả âm sạch**, ghi lại và dừng |
| **< 0.65** | nghi **lỗi triển khai** hơn kết luận khoa học ⇒ đọc lại cổng A và B |

⚠️ 1 fold (n≈80, CI ~±0.19) chỉ đủ để **LOẠI**, không đủ để **CHỌN**. Dự án đã bị cỡ mẫu nhỏ
lừa **bốn lần**.


In [ ]:
import csv as _csv
import re as _re


def _tim_run():
    trong_session = Path(os.environ["LLDMMRI_OUTPUT_DIR"])
    if list(trong_session.glob("fold*/metrics_best.json")):
        return trong_session
    for cand in sorted(Path("/kaggle/input").glob("*/**/fold*/metrics_best.json")):
        print("dùng run đã mount:", cand.parent.parent)
        return cand.parent.parent
    raise RuntimeError(
        "Không thấy fold nào có metrics_best.json. Đã tìm ở "
        + str(trong_session)
        + " và /kaggle/input/**/fold*/. `/kaggle/working` bị xoá khi session kết thúc, nên"
        " nếu bạn train ở session TRƯỚC thì phải upload thư mục run thành Kaggle Dataset rồi"
        " mount vào đây. Hoặc chạy lại cell train ở mục 2 — `resume: true` nên nó đọc tiếp"
        " từ last.pt."
    )


OUT = _tim_run()
rows = []
for d in sorted(OUT.glob("fold*")):
    mp = d / "metrics_best.json"
    if not mp.exists():
        continue
    m = _json.loads(mp.read_text("utf-8"))
    # ⚠️ Tên thư mục có HAI dạng: `fold1_<digest>` do train() sinh, và `fold_1` sau khi gói
    # lại mang về. `split("_")[0]` cho "fold" ở dạng thứ hai => int("") NỔ. Dùng regex.
    _m_fold = _re.search(r"fold_?(\d+)", d.name)
    assert _m_fold, f"không đọc được số fold từ tên thư mục {d.name!r}"
    fold = int(_m_fold.group(1))
    day = 0
    log = d / "train_log.csv"
    if log.exists():
        vals = [float(r["val_loss"]) for r in _csv.DictReader(log.open(encoding="utf-8"))]
        day = int(np.argmin(vals)) + 1 if vals else 0
    # ⚠️ `metrics_best.json` ghi **cohen_kappa**, không phải `kappa`. Đọc sai tên thì
    # `.get` trả nan và cột kappa in ra nan **im lặng**, không có cảnh báo nào.
    assert "cohen_kappa" in m, f"{mp} không có khoá cohen_kappa, chỉ có {sorted(m)}"
    rows.append(
        (fold, m["macro_f1"], m["cohen_kappa"], m.get("epoch", 0), day, m.get("per_class_f1"))
    )

assert rows, "⛔ không đọc được fold nào"

print(f"{'fold':<6}{'macro-F1':>10}{'kappa':>9}{'epoch':>7}{'val_loss đáy':>14}")
for fold, f1, kap, ep, day, _ in sorted(rows):
    print(f"{fold:<6}{f1:>10.4f}{kap:>9.4f}{ep:>7}{day:>14}")

if len(rows) > 1:
    print(f"\ntrung bình {len(rows)} fold: {float(np.mean([r[1] for r in rows])):.4f}")

# F1 từng lớp — thứ đáng đọc nhất, vì macro-F1 là trung bình của cả 7 lớp nên một lớp yếu
# kéo nó xuống nhiều hơn mức trực giác gợi ý.
from src.data.taxonomy import SHORT_NAMES

print(f"\n{'lớp':<9}" + "".join(f"{'fold ' + str(r[0]):>10}" for r in sorted(rows)))
for c in sorted(SHORT_NAMES):
    hang = "".join(
        f"{(r[5][c] if r[5] else float('nan')):>10.3f}" for r in sorted(rows)
    )
    print(f"{SHORT_NAMES[c]:<9}{hang}")

print(f"\n⚠️ ĐANG CÓ {len(rows)} FOLD.")
if len(rows) == 1:
    print("   Một fold có n≈80 ca, khoảng tin cậy 95% rộng khoảng ±0.19. Con số ở đây KHÔNG")
    print("   kết luận được gì, kể cả khi nó cao: phương sai giữa các fold của bài toán này")
    print("   lớn hơn phần lớn hiệu ứng cần đo. Chạy thêm fold trước khi tin.")
elif len(rows) < 5:
    print(f"   {len(rows)} fold đủ để LOẠI một cấu hình, chưa đủ để CHỌN nó. Một kết quả dương")
    print("   ở cỡ mẫu này chỉ có nghĩa 'chưa loại được'.")
else:
    print("   Đủ 5 fold. Con số báo cáo được là bản GỘP out-of-fold, không phải trung bình các")
    print("   fold — trung bình các fold không có khoảng tin cậy đúng nghĩa vì mỗi fold là một")
    print("   tập nhỏ khác nhau.")

print("\nHai chẩn đoán nên xem tiếp, cả hai chạy trên CPU từ xác suất đã lưu:")
print("   · tỉ lệ lỗi có biên hẹp — nói ngưỡng/hiệu chỉnh có cứu được gì không")
print("   · top-2 của các lớp yếu — nói biểu diễn có mã hoá được lớp đó không")
print("\n     python -m src.eval.run          --run-dir runs/sdrformer")
print("     python -m src.eval.weak_classes --run-dir runs/sdrformer")
print("     python -m src.eval.compare --baseline runs/Uniformer3D --candidate runs/sdrformer")

print("\n⚠️ So với cấu hình chính phải dùng `src.eval.compare` (bootstrap GHÉP CẶP trên cùng")
print("   bệnh nhân), KHÔNG so hai khoảng tin cậy rời nhau — cách đó bỏ mất phần phương sai")
print("   triệt tiêu và cho phép kiểm yếu hơn thực tế.")
print("\n⚠️ Và nhớ: phép so này gộp HAI biến (fusion và pretrained). Thấp hơn KHÔNG kết luận")
print("   được 'Siamese fusion vô ích'.")

## 4. Gói mang về

In [ ]:
import shutil

goi = Path("/kaggle/working") / f"{EXPERIMENT}_results"
if goi.exists():
    shutil.rmtree(goi)
goi.mkdir(parents=True)

for d in sorted(OUT.glob("fold*")):
    dest = goi / d.name
    dest.mkdir()
    # `train()` ghi **config_used.json**, không phải `config.yaml` — pattern cũ khớp 0 file
    # nên config lặng lẽ không được gói, và run mang về mất mất dấu vết cấu hình.
    for pattern in ("metrics_best.json", "train_log.csv", "val_probs_*.npz", "config_used.json"):
        for f in d.glob(pattern):
            shutil.copy2(f, dest / f.name)

shutil.make_archive(str(goi), "zip", goi)
print("đã gói:", goi.with_suffix(".zip"))
print("⚠️ KHÔNG gói best.pt/last.pt — checkpoint quá lớn để mang theo. Cần chúng cho suy luận")
print("   về sau thì tải riêng từ Output của session này.")
for p in sorted(goi.rglob("*")):
    print("  ", p.relative_to(goi))